# Module 13: The hooks — how it runs itself

When Claude Code "remembers you" without you asking, and when a finished conversation "files itself away" — that's **hooks**. A hook is just a command Claude Code runs automatically at a certain moment. You use two:
- **UserPromptSubmit** — runs the instant you hit enter, *before* Claude reads your message (this injects memory).
- **SessionEnd** — runs when you `/exit` (this re-ingests + re-embeds the finished session).

This module mostly *reads*; a couple of cells run.

> **Kernel:** pick **Python (memory-venv)** in the top-right kernel picker — this module imports `chromadb`/`anthropic`, which live only in that environment.

## The learning loop (7 steps)

1. **Read** the code  2. **Predict** what it does  3. **Run** it  4. **Reflect**
5. **Ask** Claude if unclear  6. **Write** a small variant  7. **Fix** a broken version

# Lesson 1: where hooks are wired — `settings.json`

**Run** to print your Claude Code settings. **Predict:** you should see both event names from the intro. What command does each one run?

In [ ]:
print(open("/Users/jenniferfletcher/.claude/settings.json").read())

**Reflect:** under `"hooks"`, each event (`UserPromptSubmit`, `SessionEnd`) lists a `command`. Both run the **venv Python** on a script — that's how Claude Code reaches your `chromadb` tools.

# Lesson 2: `memory_inject.py` — the OUT, automated

This is `semantic_search.py`'s logic, rewired to be a hook. **Run** to print it. **Predict:** how does it *receive* your message, and how does it *hand context back* to Claude?

In [ ]:
print(open("/Users/jenniferfletcher/Desktop/memory_inject.py").read())

**Reflect:** the hook contract is plumbing you already know in disguise:
- it reads your prompt from **stdin** (as JSON) — `json.loads(sys.stdin.read())`,
- it prints relevant snippets to **stdout** — and Claude Code adds whatever it prints to the conversation,
- the whole thing is wrapped in `try/except` so a memory miss can **never** crash your chat.

# Lesson 3: `refresh_memory.py` — the IN, automated

**Run** to print it. **Predict:** this file is tiny — what does it actually *do* with `ingest_jsonl.py` and `embed_messages.py`?

In [ ]:
print(open("/Users/jenniferfletcher/Desktop/refresh_memory.py").read())

**Reflect:** it uses `subprocess.run(...)` to *run other scripts* in order — first ingest the new conversation into memory.db, then embed the new messages. A script that runs scripts. SessionEnd fires it when you leave.

## ✍️ Write (step 6)

A hook reads JSON from stdin. In a notebook there's no live pipe, so **simulate it**: the variable below is exactly the shape Claude Code sends. From the blank cell, parse it and print just the prompt text.

In [ ]:
import json
fake_hook_input = '{"prompt": "tell me about my research", "session_id": "abc123"}'

# your code: turn fake_hook_input into a dict and print the value of "prompt"


---

### Now compare to a reference implementation

Below is one way to do the Write task. **Predict** what it does, run it, then compare to YOUR version. Where they differ — is one better, or are they just different styles?

In [ ]:
import json
fake_hook_input = '{"prompt": "tell me about my research", "session_id": "abc123"}'

data = json.loads(fake_hook_input)
print(data["prompt"])

# Bonus: real memory_inject.py wraps this in try/except so a malformed hook
# input can never crash your chat. Quick demonstration:
try:
    print(data["nonexistent_key"])
except KeyError as e:
    print(f"caught a KeyError silently: missing {e}")


**📝 Reflect** — how does the reference compare to yours? The bonus shows the *real* hook's defensive shape: any key error is caught and swallowed, because a memory miss must never break your chat. That's the `try/except` move from `memory_inject.py` in Lesson 2.

## 🔧 Fix (step 7)

**Broken on purpose** — it tries to read a key that doesn't exist and crashes. Read it, predict the error name, run it, then fix it so it prints the prompt. (Hint: which key actually holds the message?)

In [ ]:
import json
fake_hook_input = '{"prompt": "tell me about my research", "session_id": "abc123"}'

data = json.loads(fake_hook_input)
print(data["message"])     # <-- is "message" really the key?

## Next

Hooks are a **Claude Code** thing. Your **API client** (`chat.py`) does its own injecting and saving, no hooks involved — that's **Module 14**, the last one.